In [ ]:
# Environment and repository configuration
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo_dir = Path('/content/USD-Waymo-Capstone-Project')
    if not repo_dir.exists():
        !git clone https://github.com/bartteeuwen/USD-Waymo-Capstone-Project.git {repo_dir}
    os.chdir(repo_dir)
    !pip install -q -r requirements.txt
    !pip install -q waymo-open-dataset-tf-2-12-0 --no-deps

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

# Turning Autonomous Driving Motion Data into Interpretable Scene Intelligence
**Executive White Paper & Analytical Diagnostic Framework**  
*Bart Sosa-Teeuwen | University of San Diego*

## 1. Executive Summary & Industry Challenge
This analysis turns Waymo motion telemetry and static map metadata into interpretable scene-risk features. The detailed extraction and modeling code lives in `src/`; this notebook retains the narrative, orchestration, results, and interpretation.

## 2. Telemetry Ingestion & Physical Anomaly Screening
Scenarios with a peak speed above 38.0 m/s (~85 mph) are excluded before modeling.

In [ ]:
# Set before TensorFlow/Waymo imports in this kernel.
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from src.config import GCS_BUCKET_PATH, MAX_VELOCITY_THRESHOLD, NUM_SHARDS, TOTAL_TRAINING_SHARDS
from src.data_loader import load_and_extract_waymo_scenarios

df_raw, gnn_extracted_scenes = load_and_extract_waymo_scenarios(
    GCS_BUCKET_PATH,
    num_shards=NUM_SHARDS,
    total_shards=TOTAL_TRAINING_SHARDS,
    velocity_threshold_mps=MAX_VELOCITY_THRESHOLD,
)
print(f'Extracted {len(df_raw):,} scenarios.')

## 3. Spatial Feature Engineering & Risk Target
The target follows the original four-cluster kinematic risk definition. Spatial features are joined by `scenario_id`, so filtering invalid telemetry cannot shift features onto the wrong scene.

In [ ]:
from src.feature_engineering import DataCleaner, SpatialFeatureEngineer, assign_kinematic_risk_target

df_clean = DataCleaner(MAX_VELOCITY_THRESHOLD).clean_motion_tracks(df_raw)
df_clean = assign_kinematic_risk_target(df_clean)
df_clean = SpatialFeatureEngineer().transform(df_clean, gnn_extracted_scenes)
y = df_clean['target_risk_matrix'].map({'Standard Complexity': 0, 'Critical Complexity': 1})

display(df_clean[['max_velocity_mps', 'max_deceleration', 'min_inter_agent_dist', 'velocity_std', 'target_risk_matrix']].describe(include='all'))
print(y.value_counts().rename({0: 'Standard', 1: 'Critical'}))

## 4. Interpretable Baseline Models
The baseline uses static map friction and actor counts; the expanded models add spatial interaction metrics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from src.config import EXPANDED_FEATURES, RANDOM_STATE, TEST_SIZE
from src.modeling import initialize_tabular_models

X = df_clean[EXPANDED_FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
xgb_model, lgb_model = initialize_tabular_models(X_train, y_train, X_test, y_test)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print(f'Random Forest accuracy: {accuracy_score(y_test, rf_predictions):.4f}')
print(f'Random Forest ROC-AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.4f}')

## 5. Graph Attention Architecture
The reusable architecture takes the seven node features used in the original graph pipeline: actor type, velocity, roadgraph count, and four map-friction counts.

In [ ]:
import torch
from src.modeling import GraphAttentionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gat_model = GraphAttentionNet(in_channels=7).to(device)
print(f'Graph Attention Network initialized on {device}.')

## 6. Before-and-After Fine-Tuning
This section preserves the original analysis design: the same randomized-search grids tune Random Forest, XGBoost, and LightGBM by ROC-AUC. GCN tuning evaluates the four original dropout, width, learning-rate, and weight-decay settings. Run this only after the baseline cells; it can take several minutes.

In [ ]:
from src.tuning import evaluate_classifier, tune_tabular_models

baseline_models = {'Random Forest': rf_model, 'XGBoost': xgb_model, 'LightGBM': lgb_model}
before_tuning = {name: evaluate_classifier(model, X_test, y_test) for name, model in baseline_models.items()}
tuned_models = tune_tabular_models(X_train, y_train)
after_tuning = {name: evaluate_classifier(model, X_test, y_test) for name, model in tuned_models.items()}

comparison = pd.concat([
    pd.DataFrame(before_tuning).T.assign(Stage='Before tuning'),
    pd.DataFrame(after_tuning).T.assign(Stage='After tuning'),
]).reset_index(names='Model Family')
display(comparison[['Stage', 'Model Family', 'Test Acc', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].sort_values(['Stage', 'Model Family']))

### Tuned Spatial GCN
This reproduces the original baseline GCN and its four-setting tuned counterpart. The held-out graph split is fixed with the same random seed as the tabular comparison.

In [ ]:
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch_geometric.loader import DataLoader
from src.modeling import SpatialGCN, build_scene_graphs
from src.tuning import tune_gcn

graph_dataset = build_scene_graphs(df_clean, gnn_extracted_scenes, y)
graph_targets = [graph.y.item() for graph in graph_dataset]
train_graphs, test_graphs = train_test_split(graph_dataset, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=graph_targets)
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

baseline_gcn = SpatialGCN(in_channels=7).to(device)
optimizer = torch.optim.Adam(baseline_gcn.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()
baseline_gcn.train()
for _ in range(30):
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(baseline_gcn(batch.x, batch.edge_index, batch.batch), batch.y)
        loss.backward()
        optimizer.step()

def score_gcn(model):
    model.eval(); targets, predictions, probabilities = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            output = model(batch.x, batch.edge_index, batch.batch)
            targets.extend(batch.y.cpu().numpy())
            predictions.extend(output.argmax(dim=1).cpu().numpy())
            probabilities.extend(F.softmax(output, dim=1)[:, 1].cpu().numpy())
    return {'Test Acc': accuracy_score(targets, predictions), 'Precision': precision_score(targets, predictions, zero_division=0),
            'Recall': recall_score(targets, predictions, zero_division=0), 'F1-Score': f1_score(targets, predictions, zero_division=0),
            'ROC-AUC': roc_auc_score(targets, probabilities)}

gcn_before = score_gcn(baseline_gcn)
best_gcn = tune_gcn(train_loader, test_loader, device)
gcn_comparison = pd.DataFrame([{'Stage': 'Before tuning', **gcn_before}, {'Stage': 'After tuning', **best_gcn['metrics']}])
display(gcn_comparison)
print(f"Best GCN parameters: {best_gcn['params']}")